# MNIST Digit Classifier with PyTorch

In [ ]:
# Install required packages
%pip install torch
%pip install torchvision
%pip install numpy
%pip install matplotlib

In [2]:
# Import required packages
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [4]:
# Parameters
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# Data Pipeline
# transform converts images to PyTorch tensors and normalizes pixel values to [-1,1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load MNIST dataset
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

100.0%
100.0%
100.0%
100.0%


In [6]:
# Model Architecture
class DigitClassifierCNN(nn.Module):
    def __init__(self):
        super(DigitClassifierCNN, self).__init__()
        # First convolutional layer: input channels = 1 (grayscale), output channels = 16, kernel size = 3x3
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Second convolutional layer: input channels = 16, output channels = 32, kernel size = 3x3
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        # Fully connected layers
        self.fc1 = nn.Linear(32 * 7 * 7, 128) # 7x7 is the size after two pooling layers (28/2/2)
        self.fc2 = nn.Linear(128, 10) # 10 output classes for digits 0-9

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7) # Flatten for the fully connected layer
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [7]:
# Initialize model
model = DigitClassifierCNN().to(DEVICE)

In [8]:
# Loss Function & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [10]:
# Traning Loop
print("Starting training...") # Visualize user what is happening
for epoch in range(EPOCHS):
    model.train() # Set model to training mode
    running_loss = 0.0
    for img, label in train_loader:
        img, label = img.to(DEVICE), label.to(DEVICE)

        optimizer.zero_grad()   # Clear old gradiants
        outputs = model(img)      # Forward pass
        loss = criterion(outputs, label) # Calculate loss
        loss.backward()         # Backward pass
        optimizer.step()        # Update weights

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {running_loss/len(train_loader):.4f}")

Starting training...
Epoch [1/5], Loss: 0.1830
Epoch [2/5], Loss: 0.0530
Epoch [3/5], Loss: 0.0375
Epoch [4/5], Loss: 0.0294
Epoch [5/5], Loss: 0.0222


In [11]:
# Evaluation
model.eval() # Set model to evaluation mode
correct = 0
total = 0

with torch.no_grad(): # No need to calculate gradients during evaluation
    for img, label in test_loader:
        img, label = img.to(DEVICE), label.to(DEVICE)
        outputs = model(img)
        _, predicted = torch.max(outputs.data, 1) # Get the index of the max log-probability
        total += label.size(0)
        correct += (predicted == label).sum().item()

print(f"Accuracy: {100 * correct / total:.2f}%")

Accuracy: 98.74%
